# 05 - Ensemble Model (Walk-forward + Rolling Window)

This notebook trains and evaluates the stacking ensemble under a **walk-forward, rolling-window** protocol (`window=126`) to match the updated backtesting design.

Pipeline summary:
1. Load base-model predictions for 2024 (validation) and 2025 (test).
2. Run **Phase 1** meta-learning on 2024 using sequential walk-forward updates.
3. Run **Phase 2** forecasting on 2025 with rolling meta-learner refits and one-step-ahead predictions.
4. Compute metrics, generate comparison plots, and export final results.

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import json
from pathlib import Path

from src.utils.data_loader import load_processed_data
from src.utils.metrics import calculate_metrics
from src.models.ensemble import EnsembleModel
from src.visualization.plotter import Plotter
from src.config import RESULTS_DIR, MODELS_SAVED_DIR

plotter = Plotter()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# ── Load data ────────────────────────────────────────────────────────────────
train = load_processed_data('train')
val   = load_processed_data('val')
test  = load_processed_data('test')

target_col = 'Close'
y_val  = val[target_col].values
y_test = test[target_col].values

In [ ]:
# ── 1. Load Data & Walk-Forward Meta-Learning ───────────────────────────────
import pandas as pd
from src.config import RESULTS_DIR
from src.utils.metrics import calculate_metrics
from src.visualization.plotter import Plotter
from src.models.ensemble import EnsembleModel

plotter = Plotter()

# 1) Load base-model predictions
df_val_base = pd.read_csv(RESULTS_DIR / 'baseline_val_preds.csv', index_col=0, parse_dates=True)
df_val_lstm = pd.read_csv(RESULTS_DIR / 'lstm_val_preds.csv', index_col=0, parse_dates=True)
df_val_lstm.columns = ['LSTM']

df_test_base = pd.read_csv(RESULTS_DIR / 'baseline_test_preds.csv', index_col=0, parse_dates=True)
df_test_lstm = pd.read_csv(RESULTS_DIR / 'lstm_test_preds.csv', index_col=0, parse_dates=True)
df_test_lstm.columns = ['LSTM']

# 2) Build aligned meta-features
X_meta_train = df_val_base.join(df_val_lstm).dropna()
y_meta_train = val.loc[X_meta_train.index, 'Close']

X_meta_test = df_test_base.join(df_test_lstm).dropna()
y_true_2025 = test.loc[X_meta_test.index, 'Close']

# 3) Walk-forward + rolling-window ensemble
print("--- Training Ensemble with Walk-forward + Rolling Window (126) ---")
ensemble = EnsembleModel(alpha=1.0)

val_pred_dict = {col: X_meta_train[col].values for col in X_meta_train.columns}
test_pred_dict = {col: X_meta_test[col].values for col in X_meta_test.columns}

ens_val_series, ens_test_series = ensemble.train_and_refit_walk_forward(
    val_predictions=val_pred_dict,
    y_val=y_meta_train.values,
    test_predictions=test_pred_dict,
    y_test=y_true_2025.values,
    window=126,
    min_train_size=max(20, len(X_meta_train.columns) * 3),
)

print("\n📊 [Phase 1] 2024 Validation Metrics (Walk-forward Ensemble):")
print(calculate_metrics(y_meta_train, ens_val_series))

print("\n📊 [Phase 2] 2025 Test Metrics (Walk-forward Ensemble):")
print(calculate_metrics(y_true_2025, ens_test_series))

# Optional diagnostic: latest rolling coefficients
try:
    print("\n🔍 Latest Rolling Ensemble Weights:")
    print(ensemble.get_weights().sort_values(ascending=False))
except Exception as exc:
    print(f"Could not fetch rolling weights: {exc}")

# 4) Collect all model predictions for side-by-side comparison
all_pred_dict = {col: X_meta_test[col] for col in X_meta_test.columns}
all_pred_dict['Ensemble'] = ens_test_series

all_metrics = {}
print("\n🏆 ULTIMATE 2025 TEST SET METRICS 🏆")
for name, preds in all_pred_dict.items():
    metrics = calculate_metrics(y_true_2025, preds)
    all_metrics[name] = metrics
    print(f"\n[{name}]")
    print(f"MSE:  {metrics['mse']:.2f} | RMSE: {metrics['rmse']:.2f}")
    print(f"MAE:  {metrics['mae']:.2f} | MAPE: {metrics['mape']:.2f}%")
    print(f"Dir Acc: {metrics['directional_accuracy'] * 100:.2f}%")

# ── 3. Final Visualisations ──────────────────────────────────────────────────
plotter.plot_predictions_comparison(
    y_true=y_true_2025,
    predictions=all_pred_dict,
    dates=X_meta_test.index,
    title='2025 Final Showdown: All Base Models vs Ensemble',
    filename='ultimate_ensemble_comparison_2025.png',
)

plotter.plot_metrics_comparison(all_metrics, filename='metrics_bar_chart_2025.png')
print('All final plots generated and saved to reports/figures/ !')

# ── 4. Save the Ultimate Metrics ─────────────────────────────────────────────
final_metrics_df = pd.DataFrame(all_metrics).T
final_metrics_path = RESULTS_DIR / 'ultimate_2025_metrics.csv'
final_metrics_df.to_csv(final_metrics_path)

print(f"\n✅ Fresh metrics successfully saved to {final_metrics_path} !")
print(final_metrics_df)

# ── 5. Save the Ultimate Predictions Data (For Plotting) ───────────────────
final_preds_export = {'Actual_Close': y_true_2025}
final_preds_export.update(all_pred_dict)

final_preds_df = pd.DataFrame(final_preds_export)
final_preds_path = RESULTS_DIR / 'ultimate_2025_predictions.csv'
final_preds_df.to_csv(final_preds_path)

print(f"✅ All 2025 final predictions successfully saved to {final_preds_path} !")
print("\nPreview of first 5 rows:")
print(final_preds_df.head())

--- Training Ensemble Meta-Learner on 2024 Validation Data ---

🔍 Learned Ensemble Weights:
XGBoost    1.574780
LSTM       1.364909
Prophet    0.064801
ARIMA     -1.603494
dtype: float64

🏆 ULTIMATE 2025 TEST SET METRICS 🏆

[ARIMA]
MSE:  7133501.70 | RMSE: 2670.86
MAE:  2251.83 | MAPE: 9.66%
Dir Acc: 11.76%

[Prophet]
MSE:  3300785.88 | RMSE: 1816.81
MAE:  1356.44 | MAPE: 6.54%
Dir Acc: 59.24%

[XGBoost]
MSE:  14661710.72 | RMSE: 3829.06
MAE:  3274.12 | MAPE: 13.76%
Dir Acc: 47.06%

[LSTM]
MSE:  1576289.54 | RMSE: 1255.50
MAE:  1057.59 | MAPE: 4.49%
Dir Acc: 51.26%

[Ensemble]
MSE:  3177949.18 | RMSE: 1782.68
MAE:  1672.07 | MAPE: 7.48%
Dir Acc: 47.48%
All final plots generated and saved to reports/figures/ !

✅ Fresh metrics successfully saved to /Users/chenshuliu/Desktop/2_2_5152 ADA/2. Project/COMP5152ADA_Project_2/reports/results/ultimate_2025_metrics.csv !
                   mse         rmse          mae       mape  \
ARIMA     7.133502e+06  2670.861602  2251.833647   9.658365   


## Final Summary

| Metric | Best Model |
|--------|------------|
| RMSE   | See `reports/results/all_models_metrics.csv` |
| MAPE   | See `reports/results/all_models_metrics.csv` |
| Dir. Acc. | See `reports/results/all_models_metrics.csv` |

All results saved under `reports/results/`.